# EWC 2026 PUBG rank prediction

Self-contained notebook (no local package dependency) for Kaggle. Crawls Twire tournament stats + power rankings, scrapes Liquipedia EWC 2026 rosters, engineers weighted team features, and trains a RandomForest to predict EWC finish rank.

In [217]:
!pip install -q requests beautifulsoup4 tqdm scikit-learn pandas numpy

zsh:1: command not found: pip


In [218]:
import os
import re
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", None)

## Config

In [219]:
# --- Twire GraphQL API --------------------------------------------------------

API_URL = "https://tjjkdyimqrb7jjnc6m5rpefjtu.appsync-api.eu-west-1.amazonaws.com/graphql"

HEADERS = {
    "x-api-key": "da2-vqpq6wms5ndbvhl2r7kvzbpmfi",
    "Content-Type": "application/json",
    "Origin": "https://twire.gg",
    "Referer": "https://twire.gg/",
}

GAME = "pubg"

In [220]:
# --- Twire static assets (S3) -------------------------------------------------

TEAM_RANKING_URL = "https://twire-assets.s3.eu-west-1.amazonaws.com/pubg/team-ranking/team-ranking.json"
POWER_RANKING_URL = "https://twire-assets.s3.eu-west-1.amazonaws.com/pubg/power-ranking/power-ranking.json"
POWER_RANKING_YEAR = "2026"

# Power-ranking tournament tiers: keyed by substring of the ranking's
# tournament slug (e.g. "2026-pgs-1", "pcl-26-spring", "enc-player-ranking").
POWER_TOURNAMENT_WEIGHTS = {
    "pgs": 1.0,
    "enc": 1.0,
}
POWER_TOURNAMENT_DEFAULT_WEIGHT = 0.6  # regional: pms, pec, pvs, pas, pts, pws, pcl


In [221]:
# --- Liquipedia ----------------------------------------------------------------

LIQUIPEDIA_URL = "https://liquipedia.net/pubg/Esports_World_Cup/2026"

LIQUIPEDIA_HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

TEAM_NAME_MAP = {
    "AGAL International": "Anyone's Legend",
    "GAM The Expendables": "The Expendables",
    "FULL SENSE": "Full Sense",
    "R8 Esports": "VEGA ESPORTS",
    "Shadow Esport": "SHADOW ESPORT",
    "Sharper Esports": "Sharper Esport",
    "Team Nemesis": "TEAM NEMESIS"
}

In [222]:
# --- Tournaments -----------------------------------------------------------------

TOURNAMENTS = {
    "PGS1": {
        "id": 2473,  # Replace with actual tournament ID from tournaments.csv
        "name": "2026 PGS 1",
        "year": 2026,
        "uuid": "66bbb080-2224-11f1-8444-6adabec81c44",
    },
    "PGS2": {
        "id": 2488,  # Replace with actual ID
        "name": "2026 PGS 2",
        "year": 2025,
        "uuid": "4702993a-28f4-11f1-9125-6adabec81c44",
    },
    "PGS3": {
        "id": 2500,  # Replace with actual ID
        "name": "2026 PGS 3",
        "year": 2025,
        "uuid": "13d20004-2e72-11f1-9dc9-6adabec81c44",
    },
    "PGS4": {
        "id": 2543,  # Replace with actual ID
        "name": "2026 PGS 4",
        "year": 2025,
        "uuid": "31054c04-540a-11f1-af22-6adabec81c44",
    },
    "PGS5": {
        "id": 2551,
        "name": "2026 PGS 5",
        "year": 2026,
        "uuid": "9c951222-5a2e-11f1-9004-6adabec81c44",
    },
    "PGS6": {
        "id": 2550,  # Replace with actual tournament ID from tournaments.csv
        "name": "2026 PGS 6",
        "year": 2026,
        "uuid": "9c702f34-5a2e-11f1-8257-6adabec81c44",
    },
    "PWS": {
        "id": 2538,
        "name": "PUBG WEEKLY SERIES 2026 Phase 1",
        "year": 2026,
        "uuid": "897c012e-4b0a-11f1-8f54-6adabec81c44",
    },
    "PCL": {
        "id": 2536,
        "name": "PUBG Champions League 2026 - Spring",
        "year": 2026,
        "uuid": "8c1c3332-4a3c-11f1-a2c0-6adabec81c44",
    },
    "PEC": {
        "id": 2512,
        "name": "PEC: Spring Playoffs & Finals",
        "year": 2026,
        "uuid": "709f733c-3a81-11f1-8bfc-6adabec81c44",
    },
    "PTS": {
        "id": 2530,
        "name": "PUBG Thailand Series 2026 - Phase 1",
        "year": 2026,
        "uuid": "513a00d2-42ee-11f1-bab5-6adabec81c44",
    },
    "PAS": {
        "id": 2513,
        "name": "PAS1 Playoffs & Finals",
        "year": 2026,
        "uuid": "c2e622ae-3aad-11f1-93de-6adabec81c44",
    },
    "PVS": {
        "id": 2510,
        "name": "PVS 2026 Phase 1",
        "year": 2026,
        "uuid": "afc03272-3810-11f1-b6ee-6adabec81c44",
    },
    "PMS": {
        "id": 2400,
        "name": "PUBG Master Series 2026: Phase 1",
        "year": 2026,
        "uuid": "ec23e36e-dc10-11f0-85d2-064f26ad4164",
    },
}

In [223]:
# Tournament importance, keyed by each tournament's actual `name` (matches
# team_df["tournament"] / the Twire API's tournamentName) rather than its
# short code -- short codes like "PMS"/"PWS"/"PCL"/"PTS" don't appear as
# substrings of the real tournament names ("PUBG Master Series 2026: Phase 1"
# etc.), so keying on them silently fell through to the 1.0 default weight
# for those tournaments. Deriving from TOURNAMENTS[key]["name"] keeps this
# in sync automatically.
_PGS_WEIGHTS = {
    "PGS1": 0.80,
    "PGS2": 0.90,
    "PGS3": 1.00,
    "PGS4": 1.10,
    "PGS5": 1.20,
    "PGS6": 1.30,
}
_REGIONAL_DEFAULT_WEIGHT = 0.60

TOURNAMENT_WEIGHTS = {
    TOURNAMENTS[key]["name"]: _PGS_WEIGHTS.get(key, _REGIONAL_DEFAULT_WEIGHT)
    for key in TOURNAMENTS
}

# 6 individual PGS tiers (0.80-1.30) + 1 shared regional tier (0.60) = 7.
TOTAL_TOURNAMENT_TIERS = len(set(TOURNAMENT_WEIGHTS.values()))

# Stage importance
STAGE_WEIGHTS = {
    "Grand Finals": 1.30,
    "Final Stage": 1.30,
    "Winners Stage": 1.15,
    "Group Stage": 1.00,
    "Survival Stage": 0.95,
}


In [224]:
# --- Output --------------------------------------------------------------------
# "../output" so it lands in the project-root output/ dir, not notebooks/output/
# (Jupyter's cwd is wherever this notebook file lives).

OUTPUT_DIR = "../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Normalization & weighting helpers

In [225]:
def get_tournament_weight(name):
    for key, weight in TOURNAMENT_WEIGHTS.items():
        if key.lower() in str(name).lower():
            return weight
    return 1.0


def get_stage_weight(name):
    for key, weight in STAGE_WEIGHTS.items():
        if key.lower() in str(name).lower():
            return weight
    return 0.6


def get_power_tournament_weight(tournament):
    for key, weight in POWER_TOURNAMENT_WEIGHTS.items():
        if key.lower() in str(tournament).lower():
            return weight
    return POWER_TOURNAMENT_DEFAULT_WEIGHT


In [226]:
def normalize_team(name):
    if pd.isna(name):
        return name

    name = str(name).strip()

    # remove duplicate spaces
    name = re.sub(r"\s+", " ", name)

    # remove spaces in names beginning with numbers
    # 17 Gaming -> 17Gaming
    name = re.sub(r"^(\d+)\s+", r"\1", name)

    return name


def normalize_player(name):
    if pd.isna(name):
        return name

    name = str(name).strip()

    # Take everything after the first underscore
    if "_" in name:
        name = name.split("_", 1)[1]

    return name

## 1. Twire tournament stats (GraphQL)

In [227]:
GET_FILTERS = """
query ($id: Int!, $game: String!) {
  tournamentInitialData(id: $id, game: $game) {
    tournament {
      friendlyName
    }
    tournamentFilters {
      name
      value
    }
  }
}
"""

TEAM_STATS_QUERY = """
query ($shardInfo:String!, $game:String!) {
  teamStats(
    shardInfo:$shardInfo,
    token:"",
    filters:null,
    game:$game
  ) {

    tournamentName
    groupName
    matchName

    teamStats {

      teamName
      teamLogo

      stats {

        map
        rank
        kills
        assists
        damageDealt
        damageTaken
        points
        totalPoints
        wins
        avgKills
        avgDamageDealt
        avgRank
        avgPoints
      }
    }
  }
}
"""

PLAYER_STATS_QUERY = """
query ($shardInfo:String!, $game:String!) {
  platformStats(
    tournament:$shardInfo,
    token:"",
    filters:null,
    game:$game
  ) {

    tournamentName
    groupName
    matchName

    leaderboard {

      username
      teamName

      kills
      assists

      kd
      kas

      damageDealt
      damageTaken

      avgDamageDealt
      avgDamageTaken

      deaths

      dbnos

      revives

      headshotKills

      walkDistance
      rideDistance

      longestKill

      avgTimeSurvived

      numOfMatches
    }
  }
}
"""

In [228]:
def graphql(query, variables=None):
    payload = {
        "query": query,
        "variables": variables or {}
    }

    r = requests.post(API_URL, headers=HEADERS, json=payload)

    print("Status:", r.status_code)

    try:
        return r.json()
    except Exception:
        print(r.text)
        return None

In [229]:
def fetch_shard_infos(tournaments):
    """Resolve each tournament's stage filters into GraphQL shardInfo strings."""

    all_filters = {}

    for key, t in tournaments.items():

        result = graphql(GET_FILTERS, {"id": t["id"], "game": GAME})

        if result is None or "errors" in result:
            print("❌ Error")
            continue

        if result["data"]["tournamentInitialData"] is None:
            print("❌ No tournamentInitialData")
            continue

        all_filters[key] = result["data"]["tournamentInitialData"]["tournamentFilters"]

    shard_infos = {}

    for tournament, filters in all_filters.items():

        uuid = tournaments[tournament]["uuid"]

        shard_infos[tournament] = []

        for f in filters:

            shard_infos[tournament].append({
                "stage": f["name"],
                "value": f["value"],
                "shardInfo": f"{uuid}-{f['value']}"
            })

    return shard_infos

In [230]:
shard_infos = fetch_shard_infos(TOURNAMENTS)
shard_infos.keys()

Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200


dict_keys(['PGS1', 'PGS2', 'PGS3', 'PGS4', 'PGS5', 'PGS6', 'PWS', 'PCL', 'PEC', 'PTS', 'PAS', 'PVS', 'PMS'])

In [231]:
def fetch_team_stats_raw(shard_infos):
    team_stats_raw = []

    for tournament, stages in tqdm(shard_infos.items()):

        for stage in stages:

            data = graphql(
                TEAM_STATS_QUERY,
                {
                    "shardInfo": stage["shardInfo"],
                    "game": GAME
                }
            )

            team_stats_raw.append({
                "tournament": tournament,
                "stage": stage["stage"],
                "data": data["data"]["teamStats"]
            })

    return team_stats_raw


team_stats_raw = fetch_team_stats_raw(shard_infos)
len(team_stats_raw)

  0%|          | 0/13 [00:00<?, ?it/s]

Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200


  8%|▊         | 1/13 [00:04<00:49,  4.13s/it]

Status: 200
Status: 200
Status: 200


 15%|█▌        | 2/13 [00:07<00:37,  3.41s/it]

...

 92%|█████████▏| 12/13 [00:56<00:04,  4.70s/it]

Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200


100%|██████████| 13/13 [01:11<00:00,  5.52s/it]

Status: 200


100

In [232]:
def fetch_player_stats_raw(shard_infos):
    player_stats_raw = []

    for tournament, stages in tqdm(shard_infos.items()):

        for stage in stages:

            data = graphql(
                PLAYER_STATS_QUERY,
                {
                    "shardInfo": stage["shardInfo"],
                    "game": GAME
                }
            )

            player_stats_raw.append({
                "tournament": tournament,
                "stage": stage["stage"],
                "data": data["data"]["platformStats"]
            })

    return player_stats_raw


player_stats_raw = fetch_player_stats_raw(shard_infos)
len(player_stats_raw)

  0%|          | 0/13 [00:00<?, ?it/s]

Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200


  8%|▊         | 1/13 [00:05<01:07,  5.60s/it]

Status: 200
Status: 200
Status: 200


 15%|█▌        | 2/13 [00:08<00:41,  3.74s/it]

...

 92%|█████████▏| 12/13 [01:03<00:04,  4.74s/it]

Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200
Status: 200


100%|██████████| 13/13 [01:21<00:00,  6.26s/it]

Status: 200


100

In [233]:
def build_team_df(team_stats_raw):
    rows = []

    for tournament in team_stats_raw:

        info = tournament["data"]

        for team in info["teamStats"]:

            for stat in team["stats"]:

                row = {
                    "tournament": info["tournamentName"],
                    "stage": info["groupName"],
                    "team": team["teamName"],
                    **stat
                }

                rows.append(row)

    return pd.DataFrame(rows)


team_df = build_team_df(team_stats_raw)
team_df.shape

(7966, 16)

In [234]:
def build_player_df(player_stats_raw):
    rows = []

    for tournament in player_stats_raw:

        info = tournament["data"]

        for player in info["leaderboard"]:

            player["tournament"] = info["tournamentName"]
            player["stage"] = info["groupName"]

            rows.append(player)

    return pd.DataFrame(rows)


player_df = build_player_df(player_stats_raw)
player_df.shape

(6647, 21)

In [235]:
def apply_weights(df):
    """Attach tournament/stage importance weights (used by all weighted averages)."""

    df["tournament_weight"] = df["tournament"].apply(get_tournament_weight)
    df["stage_weight"] = df["stage"].apply(get_stage_weight)
    df["weight"] = (
        df["tournament_weight"]
        * df["stage_weight"]
    )
    return df


team_df = apply_weights(team_df)
player_df = apply_weights(player_df)

team_df[["tournament", "stage", "weight"]].drop_duplicates().sort_values(["tournament", "stage"])

,tournament,stage,weight
0,2026 PGS 1,Final Stage Grand Finals,1.04
464,2026 PGS 1,Group Stage Group A x B,0.80
336,2026 PGS 1,Group Stage Group A x C,0.80
400,2026 PGS 1,Group Stage Group B x C,0.80
240,2026 PGS 1,Group Stage Overall,0.80
...,...,...,...
2216,PUBG WEEKLY SERIES 2026 Phase 1,Weekly Stage 3 Group A x B,0.36
2136,PUBG WEEKLY SERIES 2026 Phase 1,Weekly Stage 3 Group A x C,0.36
2056,PUBG WEEKLY SERIES 2026 Phase 1,Weekly Stage 3 Group B x C,0.36
1936,PUBG WEEKLY SERIES 2026 Phase 1,Weekly Stage 3 Overall,0.36


In [236]:
team_df.to_csv(f"{OUTPUT_DIR}/team_stats.csv", index=False)
player_df.to_csv(f"{OUTPUT_DIR}/player_stats.csv", index=False)

print(team_df.shape)
print(player_df.shape)

(7966, 19)
(6647, 24)


## 2. Twire team/power rankings (S3)

In [237]:
def fetch_team_ranking():
    data = requests.get(TEAM_RANKING_URL).json()["teams"]
    return pd.DataFrame(data)


team_rank_df = fetch_team_ranking()  # not used downstream, kept for parity with the original notebook
team_rank_df.head()

,name,ranking,id,previousPosition,previousPositionChange
0,Virtus.pro,799,3698,SAME,0
1,Twisted Minds,721,24823,SAME,0
2,Full Sense,614,12564,SAME,0
3,Team Falcons,594,30472,SAME,0
4,DN FREECS,529,41253,SAME,0


In [238]:
POWER_SCORE_COLUMNS = [
    "overall_score",
    "attacker_score",
    "survivor_score",
    "teammate_score",
    "utility_score",
    "finisher_score",
]


def fetch_power_ranking():
    power_data = requests.get(POWER_RANKING_URL).json()["ranking"][POWER_RANKING_YEAR]

    rows = []

    for tournament, tournament_data in power_data.items():

        for player in tournament_data["players"]:

            rows.append({
                "tournament": tournament,
                "team": player["team"],
                "nickname": player["nickname"],

                "overall_score": player["overallScore"],
                "attacker_score": player["attackerScore"],
                "survivor_score": player["survivorScore"],
                "teammate_score": player["teammateScore"],
                "utility_score": player["utilityScore"],
                "finisher_score": player["finisherScore"]
            })

    power_df = pd.DataFrame(rows)
    power_df["team"] = power_df["team"].apply(normalize_team)

    # Scale each player's scores against that tournament's own top score
    # (so a regional event's rating scale doesn't read as equal to a PGS's),
    # then weight by tournament tier (PGS/ENC 1.0, regional 0.6).
    power_df["tournament_weight"] = power_df["tournament"].apply(get_power_tournament_weight)

    # Weighted columns feed the rating aggregates below; the raw overall_score
    # is kept as-is since star_players thresholds against its natural 0-100 scale.
    for col in POWER_SCORE_COLUMNS:
        tournament_max = power_df.groupby("tournament")[col].transform("max")
        power_df[f"{col}_weighted"] = (power_df[col] / tournament_max) * power_df["tournament_weight"]

    return power_df


power_df = fetch_power_ranking()
power_df.head()


,tournament,team,nickname,overall_score,attacker_score,survivor_score,teammate_score,utility_score,finisher_score,tournament_weight,overall_score_weighted,attacker_score_weighted,survivor_score_weighted,teammate_score_weighted,utility_score_weighted,finisher_score_weighted
0,pms-26-phase-1,Orgless,alec,95.0,97.0,90.0,87.0,98.0,85.0,0.6,0.600000,0.600000,0.540,0.561290,0.600000,0.554348
1,pms-26-phase-1,Alter Ego,tRycK,94.0,93.0,95.0,87.0,92.0,87.0,0.6,0.593684,0.575258,0.570,0.561290,0.563265,0.567391
2,pms-26-phase-1,ZANSHIN,akosiRE,93.0,95.0,92.0,85.0,87.0,88.0,0.6,0.587368,0.587629,0.552,0.548387,0.532653,0.573913
3,pms-26-phase-1,Alter Ego,QUEEN,92.0,90.0,95.0,89.0,93.0,83.0,0.6,0.581053,0.556701,0.570,0.574194,0.569388,0.541304
4,pms-26-phase-1,VEGA ESPORTS,empt,91.0,86.0,100.0,93.0,91.0,84.0,0.6,0.574737,0.531959,0.600,0.600000,0.557143,0.547826


In [239]:
power_df[power_df["tournament"] == "enc-player-ranking"].sort_values("overall_score", ascending=False).head(10)
# power_df["tournament"].unique()

,tournament,team,nickname,overall_score,attacker_score,survivor_score,teammate_score,utility_score,finisher_score,tournament_weight,overall_score_weighted,attacker_score_weighted,survivor_score_weighted,teammate_score_weighted,utility_score_weighted,finisher_score_weighted
955,enc-player-ranking,China,xwudd,230.8,230.8,230.8,230.8,230.8,230.8,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
956,enc-player-ranking,Vietnam,Himass,230.2,230.2,230.2,230.2,230.2,230.2,1.0,0.997400,0.997400,0.997400,0.997400,0.997400,0.997400
957,enc-player-ranking,Russia,Lev4nte,227.8,227.8,227.8,227.8,227.8,227.8,1.0,0.987002,0.987002,0.987002,0.987002,0.987002,0.987002
958,enc-player-ranking,China,WenBo,225.6,225.6,225.6,225.6,225.6,225.6,1.0,0.977470,0.977470,0.977470,0.977470,0.977470,0.977470
959,enc-player-ranking,Ukraine,Hakatory,218.6,218.6,218.6,218.6,218.6,218.6,1.0,0.947140,0.947140,0.947140,0.947140,0.947140,0.947140
960,enc-player-ranking,South Korea,Pio,216.2,216.2,216.2,216.2,216.2,216.2,1.0,0.936742,0.936742,0.936742,0.936742,0.936742,0.936742
961,enc-player-ranking,China,Lu,215.6,215.6,215.6,215.6,215.6,215.6,1.0,0.934142,0.934142,0.934142,0.934142,0.934142,0.934142
962,enc-player-ranking,South Korea,Inonix,212.4,212.4,212.4,212.4,212.4,212.4,1.0,0.920277,0.920277,0.920277,0.920277,0.920277,0.920277
963,enc-player-ranking,China,Lilghost,210.6,210.6,210.6,210.6,210.6,210.6,1.0,0.912478,0.912478,0.912478,0.912478,0.912478,0.912478
965,enc-player-ranking,Russia,hallomybad,210.0,210.0,210.0,210.0,210.0,210.0,1.0,0.909879,0.909879,0.909879,0.909879,0.909879,0.909879


In [240]:
def build_team_power(power_df):
    return (
        power_df
        .groupby(["tournament", "team"])
        .agg(
            power_avg=("overall_score_weighted", "mean"),
            power_max=("overall_score_weighted", "max"),

            attacker_avg=("attacker_score_weighted", "mean"),
            survivor_avg=("survivor_score_weighted", "mean"),
            teammate_avg=("teammate_score_weighted", "mean"),
            utility_avg=("utility_score_weighted", "mean"),
            finisher_avg=("finisher_score_weighted", "mean"),

            star_players=("overall_score", lambda x: (x >= 85).sum())
        )
        .reset_index()
    )


team_power = build_team_power(power_df)
team_power


,tournament,team,power_avg,power_max,attacker_avg,survivor_avg,teammate_avg,utility_avg,finisher_avg,star_players
0,2026-pas-1,55e-Sports,0.487895,0.498947,0.457653,0.5160,0.528125,0.462121,0.477128,0
1,2026-pas-1,Affinity,0.454737,0.486316,0.425510,0.4635,0.485938,0.442424,0.457979,0
2,2026-pas-1,Also Known as,0.406316,0.416842,0.378571,0.3850,0.402083,0.373737,0.395745,0
3,2026-pas-1,BESTIA,0.498947,0.517895,0.445408,0.5610,0.571875,0.471212,0.486702,0
4,2026-pas-1,Chupinsky's,0.449684,0.511579,0.415102,0.4680,0.477500,0.426667,0.460851,0
...,...,...,...,...,...,...,...,...,...,...
276,pws-26-phase-1,Star Balloon,0.476842,0.498947,0.473196,0.4890,0.494211,0.434848,0.491753,0
277,pws-26-phase-1,T1,0.513158,0.562105,0.518041,0.4995,0.502105,0.501515,0.493299,2
278,pws-26-phase-1,Vex E-Sports,0.470526,0.511579,0.490206,0.4980,0.502105,0.431818,0.453093,0
279,pws-26-phase-1,XX Gaming,0.413684,0.435789,0.427835,0.4080,0.435789,0.405051,0.411340,0


In [241]:
def build_history_power(team_power):
    # Sum a team's (already tournament-weighted, tournament-normalized) power
    # across every tournament it appeared in, rewarding repeated strong
    # showings rather than averaging them away.
    return (
        team_power
        .groupby("team")
        .agg(
            twire_power=("power_avg", "sum"),
            twire_peak=("power_max", "sum"),
            attacker_rating=("attacker_avg", "sum"),
            survivor_rating=("survivor_avg", "sum"),
            teammate_rating=("teammate_avg", "sum"),
            utility_rating=("utility_avg", "sum"),
            finisher_rating=("finisher_avg", "sum"),
            star_players=("star_players", "sum")
        )
        .reset_index()
    )


history_power = build_history_power(team_power)
history_power


,team,twire_power,twire_peak,attacker_rating,survivor_rating,teammate_rating,utility_rating,finisher_rating,star_players
0,17Gaming,3.286026,3.482494,3.182243,3.301667,3.411463,3.212646,3.183853,9
1,55e-Sports,0.487895,0.498947,0.457653,0.516000,0.528125,0.462121,0.477128,0
2,ACEND,0.489894,0.504255,0.472674,0.525000,0.542187,0.450000,0.450000,0
3,ARETE,0.456316,0.511579,0.482474,0.456000,0.467368,0.418182,0.462371,0
4,ATO ESPORT,0.468947,0.492632,0.450000,0.442500,0.482258,0.451531,0.474457,0
...,...,...,...,...,...,...,...,...,...
206,Zhu Chao Gaming,0.454737,0.480000,0.485567,0.502500,0.510000,0.401515,0.451546,0
207,eArena,2.878179,3.007480,2.836648,3.060568,3.195540,2.841807,2.917935,3
208,exHowl,0.434043,0.453191,0.449302,0.434400,0.460000,0.396364,0.406737,0
209,noslack,0.528191,0.555319,0.500581,0.541500,0.564063,0.459091,0.506842,1


## 3. Liquipedia EWC 2026 rosters

Falls back through: cached `output/ewc2026_rosters.csv` → a manually saved copy at `data/liquipedia_ewc2026.html` → a live scrape. If you need to refresh the saved HTML, open the Liquipedia EWC 2026 page in a browser, save it as `data/liquipedia_ewc2026.html`, and delete `output/ewc2026_rosters.csv` to force a re-parse.

In [242]:
def fetch_ewc_rosters(html=None):
    """Scrape confirmed EWC 2026 rosters.

    Pass `html=` with a manually saved copy of the page if Liquipedia's
    Cloudflare challenge is blocking the automated request.
    """

    if html is None:
        html = requests.get(LIQUIPEDIA_URL, headers=LIQUIPEDIA_HEADERS).text

    soup = BeautifulSoup(html, "html.parser")
    cards = soup.select("div.team-participant-card")

    if not cards:
        raise RuntimeError(
            "No team-participant-card elements found on the Liquipedia page. "
            "The site is likely blocking the automated request (e.g. a "
            "Cloudflare challenge) rather than the page layout having "
            "changed. Try again later, or open the URL in a browser, save "
            'the rendered HTML, and pass it via fetch_ewc_rosters(html=...).'
        )

    rows = []

    for card in cards:

        team_tag = card.select_one(
            ".team-participant-card__opponent-full .name a"
        )

        team = team_tag.get_text(strip=True)

        players = [
            p.get_text(strip=True)
            for p in card.select(".block-player .name a")
        ]

        for player in players[0:4]:

            rows.append({
                "team": team,
                "player": player
            })

    players_df = pd.DataFrame(rows)

    players_df["tournament"] = "EWC2026"

    players_df = players_df[
        ["tournament", "team", "player"]
    ]

    players_df["team"] = (
        players_df["team"]
        .replace(TEAM_NAME_MAP)
    )

    return players_df

In [243]:
EWC2026_ROSTERS = [
    ("Twisted Minds", "BatulinS"),
    ("Twisted Minds", "Perfect1ks"),
    ("Twisted Minds", "xmpl"),
    ("Twisted Minds", "Lu"),
    ("Made in Thailand", "KISS"),
    ("Made in Thailand", "Jacob"),
    ("Made in Thailand", "Scappy"),
    ("Made in Thailand", "Baren"),
    ("Virtus.pro", "Lukarux"),
    ("Virtus.pro", "Beami"),
    ("Virtus.pro", "curexi"),
    ("Virtus.pro", "NIXZYEE"),
    ("17 Gaming", "Lilghost"),
    ("17 Gaming", "tiantianhaovo"),
    ("17 Gaming", "WenBo"),
    ("17 Gaming", "xwudd"),
    ("T1", "EEND"),
    ("T1", "Heather"),
    ("T1", "Rain1ng"),
    ("T1", "Type"),
    ("Natus Vincere", "Feyerist"),
    ("Natus Vincere", "Hakatory"),
    ("Natus Vincere", "boost1k-"),
    ("Natus Vincere", "spyrro"),
    ("Petrichor Road", "MMing"),
    ("Petrichor Road", "i26v6"),
    ("Petrichor Road", "Cui71"),
    ("Petrichor Road", "04NB"),
    ("SOOPers", "DIEL"),
    ("SOOPers", "Gyumin"),
    ("SOOPers", "Heaven"),
    ("SOOPers", "Rex"),
    ("Anyone's Legend", "Delwyn"),
    ("Anyone's Legend", "Himass"),
    ("Anyone's Legend", "Sololzy"),
    ("Anyone's Legend", "Destroyy"),
    ("Four Angry Men", "HSmm"),
    ("Four Angry Men", "Shen"),
    ("Four Angry Men", "SpaceMan1010"),
    ("Four Angry Men", "WINDah"),
    ("Team Falcons", "Shrimzy"),
    ("Team Falcons", "Gustav"),
    ("Team Falcons", "TGLTN"),
    ("Team Falcons", "Kickstart"),
    ("Team Liquid", "PurdyKurty"),
    ("Team Liquid", "CowBoi"),
    ("Team Liquid", "aLOW"),
    ("Team Liquid", "luke12"),
    ("JD Gaming", "Dec12th"),
    ("JD Gaming", "nanss"),
    ("JD Gaming", "Cold119"),
    ("JD Gaming", "SuZe"),
    ("TYLOO", "1ee"),
    ("TYLOO", "KKong"),
    ("TYLOO", "HaoSkr"),
    ("TYLOO", "OneDragon"),
    ("Team Vitality", "Gedrox"),
    ("Team Vitality", "hallomybad"),
    ("Team Vitality", "Lev4nte"),
    ("Team Vitality", "QWZYYY"),
    ("TEAM NEMESIS", "DIFX"),
    ("TEAM NEMESIS", "SoseD"),
    ("TEAM NEMESIS", "Staed"),
    ("TEAM NEMESIS", "Mellman"),
    ("Geekay Esports", "AKaN"),
    ("Geekay Esports", "EJ"),
    ("Geekay Esports", "Parkpro"),
    ("Geekay Esports", "Seongjang"),
    ("Gen.G Esports", "seoul"),
    ("Gen.G Esports", "Salute"),
    ("Gen.G Esports", "diyy"),
    ("Gen.G Esports", "BeaN"),
    ("Full Sense", "Thanad0l"),
    ("Full Sense", "TiGGER"),
    ("Full Sense", "Belmoth"),
    ("Full Sense", "Flash"),
    ("Sharper Esport", "ThanawatTH"),
    ("Sharper Esport", "Jdaii"),
    ("Sharper Esport", "Earthzapalui"),
    ("Sharper Esport", "Thunderz"),
    ("The Expendables", "Clories"),
    ("The Expendables", "DuCkHjeUz"),
    ("The Expendables", "TanVuu"),
    ("The Expendables", "Hoangf"),
    ("The Vicious", "YmCud"),
    ("The Vicious", "Sapauu"),
    ("The Vicious", "Taikonn"),
    ("The Vicious", "SirT"),
    ("SHADOW ESPORT", "Setsunaa"),
    ("SHADOW ESPORT", "Jekzy"),
    ("SHADOW ESPORT", "Hanzyy"),
    ("SHADOW ESPORT", "Kamalz"),
    ("VEGA ESPORTS", "AJ"),
    ("VEGA ESPORTS", "1MBOT"),
    ("VEGA ESPORTS", "Tedeeyy"),
    ("VEGA ESPORTS", "empt"),
]

roster_path = f"{OUTPUT_DIR}/ewc2026_rosters.csv"

try:
    players_df = pd.read_csv(roster_path)
    print(f"Using cached rosters: {roster_path}")
except FileNotFoundError:
    print("No cached rosters found -- using hardcoded EWC 2026 roster snapshot")
    players_df = pd.DataFrame(EWC2026_ROSTERS, columns=["team", "player"])
    players_df["tournament"] = "EWC2026"
    players_df = players_df[["tournament", "team", "player"]]

players_df

Using cached rosters: ../output/ewc2026_rosters.csv


,tournament,team,player
0,EWC2026,Twisted Minds,BatulinS
1,EWC2026,Twisted Minds,Perfect1ks
2,EWC2026,Twisted Minds,xmpl
3,EWC2026,Twisted Minds,Lu
4,EWC2026,Made in Thailand,KISS
...,...,...,...
91,EWC2026,SHADOW ESPORT,Kamalz
92,EWC2026,VEGA ESPORTS,AJ
93,EWC2026,VEGA ESPORTS,1MBOT
94,EWC2026,VEGA ESPORTS,Tedeeyy


## 4. Feature engineering

In [244]:
team_df = team_df[team_df["map"] == "all"].copy()
team_df.shape

(1622, 19)

In [245]:
def wavg(x, value, weight_col="weight"):
    return np.average(x[value], weights=x[weight_col])


def build_player_features(player_df):
    return (
        player_df
        .groupby(["tournament", "stage", "teamName"])
        .apply(lambda x: pd.Series({
            "avg_kd": wavg(x, "kd"),
            "avg_damage": wavg(x, "damageDealt"),
            "avg_kills": wavg(x, "kills"),
            "avg_assists": wavg(x, "assists"),
            "avg_kas": wavg(x, "kas"),

            "max_damage": x["damageDealt"].max(),
            "max_kd": x["kd"].max(),

            "roster_size": x["username"].nunique()
        }))
        .reset_index()
    )


player_features = build_player_features(player_df)
player_features.head()

/var/folders/fy/cpmyp83s3rb3s_xkpdyggd2m0000gn/T/ipykernel_1724/376917467.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  player_df


,tournament,stage,teamName,avg_kd,avg_damage,avg_kills,avg_assists,avg_kas,max_damage,max_kd,roster_size
0,2026 PGS 1,Final Stage Grand Finals,Anyone's Legend,1.325,2523.8450,11.75,8.25,0.825,3139.05,1.7,4.0
1,2026 PGS 1,Final Stage Grand Finals,DN soopers,0.825,1719.4475,7.50,4.25,0.600,2448.89,1.2,4.0
2,2026 PGS 1,Final Stage Grand Finals,FURIA,0.550,1480.9775,5.50,3.25,0.375,2201.86,1.0,4.0
3,2026 PGS 1,Final Stage Grand Finals,Finhay Cerberus,1.150,1841.7400,11.00,6.00,0.650,2066.04,1.3,4.0
4,2026 PGS 1,Final Stage Grand Finals,Four Angry Men,1.525,2036.1400,14.25,5.75,0.700,2507.82,2.3,4.0


In [246]:
def build_dataset(team_df, player_features):
    return team_df.merge(
        player_features,
        left_on=["tournament", "stage", "team"],
        right_on=["tournament", "stage", "teamName"],
        how="left"
    )


dataset = build_dataset(team_df, player_features)
dataset.shape

(1622, 28)

In [247]:
# Apply to every dataframe
for df, cols in [
    (team_df, ["team"]),
    (player_df, ["teamName"]),
    (players_df, ["team"]),
]:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].apply(normalize_team)

In [248]:
player_df["username"] = player_df["username"].apply(normalize_player)
players_df["player"] = players_df["player"].apply(normalize_player)

In [249]:
ewc_teams = players_df["team"].unique().tolist()
print(f"{len(ewc_teams)} EWC teams")
ewc_teams

24 EWC teams


['Twisted Minds',
 'Made in Thailand',
 'Virtus.pro',
 '17Gaming',
 'T1',
 'Natus Vincere',
 'Petrichor Road',
 'SOOPers',
 "Anyone's Legend",
 'Four Angry Men',
 'Team Falcons',
 'Team Liquid',
 'JD Gaming',
 'TYLOO',
 'Team Vitality',
 'TEAM NEMESIS',
 'Geekay Esports',
 'Gen.G Esports',
 'Full Sense',
 'Sharper Esport',
 'The Expendables',
 'The Vicious',
 'SHADOW ESPORT',
 'VEGA ESPORTS']

In [250]:
def build_history(dataset, ewc_teams):
    return dataset[
        dataset["team"].isin(ewc_teams)
    ].copy()


history = build_history(dataset, ewc_teams)
history.shape

(407, 28)

In [251]:
def _team_history_row(x):
    # A team's own average tournament tier (PGS ~0.8-1.3, regional 0.6).
    # np.average(..., weights=x["weight"]) alone only weighs a team's
    # tournaments *relative to each other* -- it cancels out for a team
    # that only ever played one tier, so a regional-only team's stats
    # would otherwise come out on the same scale as a globals team's.
    tier = np.average(x["tournament_weight"], weights=x["weight"])

    # Breadth: how many of the 7 recognized tiers (6 individual PGS majors
    # + 1 shared regional tier) has this team actually competed in. A team
    # that's only ever played one regional event gets hit twice: once for
    # low tier weight, again for having proven nothing outside that one
    # tier -- compounding into a much steeper discount than tier alone.
    coverage = x["tournament_weight"].nunique() / TOTAL_TOURNAMENT_TIERS

    discount = tier * coverage

    return pd.Series({

        "tournaments": x["tournament"].nunique(),
        "stages": len(x),

        "avg_rank": np.average(x["avgRank"], weights=x["weight"]),
        "std_rank": x["avgRank"].std(),

        "avg_points": np.average(x["totalPoints"], weights=x["weight"]) * discount,
        "std_points": x["totalPoints"].std(),

        "avg_kills": np.average(x["kills"], weights=x["weight"]) * discount,
        "std_kills": x["kills"].std(),

        "avg_damage": np.average(x["damageDealt"], weights=x["weight"]) * discount,
        "std_damage": x["damageDealt"].std(),

        "avg_damage_taken": np.average(x["damageTaken"], weights=x["weight"]) * discount,

        "total_wins": (x["wins"] * x["weight"]).sum(),

        "player_avg_kd": np.average(x["avg_kd"], weights=x["weight"]) * discount,
        "player_max_kd": np.average(x["max_kd"], weights=x["weight"]) * discount,

        "player_avg_damage": np.average(x["avg_damage"], weights=x["weight"]) * discount,
        "player_max_damage": np.average(x["max_damage"], weights=x["weight"]) * discount,

        "player_avg_kills": np.average(x["avg_kills"], weights=x["weight"]) * discount,
        "player_avg_assists": np.average(x["avg_assists"], weights=x["weight"]) * discount,

        "player_avg_kas": np.average(x["avg_kas"], weights=x["weight"]) * discount,

    })


def build_history_features(history, history_power):
    history_features = (
        history
        .groupby("team")
        .apply(_team_history_row)
        .reset_index()
    )

    history_features = history_features.merge(
        history_power,
        on="team",
        how="left"
    )

    history_features = history_features.fillna(0)

    return history_features


history_features = build_history_features(history, history_power)
history_features.head()


/var/folders/fy/cpmyp83s3rb3s_xkpdyggd2m0000gn/T/ipykernel_1724/492716213.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  history


,team,tournaments,stages,avg_rank,std_rank,avg_points,std_points,avg_kills,std_kills,avg_damage,std_damage,avg_damage_taken,total_wins,player_avg_kd,player_max_kd,player_avg_damage,player_max_damage,player_avg_kills,player_avg_assists,player_avg_kas,twire_power,twire_peak,attacker_rating,survivor_rating,teammate_rating,utility_rating,finisher_rating,star_players
0,17Gaming,6.0,17.0,7.151294,2.321780,46.326559,30.421548,28.844480,20.290174,5169.961846,3158.719594,5696.224908,16.535,1.407473,2.178229,1295.051758,1683.827757,7.211120,4.001565,0.611828,3.286026,3.482494,3.182243,3.301667,3.411463,3.212646,3.183853,9
1,Anyone's Legend,6.0,17.0,7.978552,1.479467,59.122781,141.912333,38.360459,94.963189,6896.122852,14882.523217,8323.178440,17.480,0.953857,1.535238,1727.382471,2482.970162,9.590115,5.171359,0.539392,3.051851,3.410526,2.987854,3.068273,3.191376,3.023485,3.016464,5
2,Four Angry Men,7.0,26.0,7.554494,1.913358,50.496167,41.509684,32.961037,29.939246,6155.619570,5337.346959,7100.901476,11.175,1.260614,2.107765,1542.081718,1998.964967,8.240259,4.379935,0.622854,2.723608,2.877048,2.745554,2.811939,2.929191,2.750779,2.742652,0
3,Full Sense,7.0,23.0,8.286147,2.309355,42.887000,65.618643,29.180892,40.902226,5403.816728,6529.200101,6245.663502,8.970,1.034778,1.576334,1295.777324,1782.796541,6.934284,4.293360,0.625837,1.925987,2.094737,1.910194,1.904250,1.984037,1.832735,1.872168,4
4,Geekay Esports,1.0,13.0,6.945882,1.508778,6.018151,34.859609,3.866218,22.157073,655.273613,3608.952679,745.778824,4.740,0.116647,0.157008,164.153995,203.649872,0.966555,0.523866,0.062947,0.566842,0.600000,0.559794,0.552000,0.541579,0.539394,0.547423,3


In [252]:
history_features.to_csv(f"{OUTPUT_DIR}/ewc_history_features.csv", index=False)

## 5. Model

In [253]:
FEATURES = [
    "tournaments",
    "stages",

    "avg_points",
    "std_points",

    # "avg_rank",
    "std_rank",

    "avg_kills",
    "std_kills",

    "avg_damage",
    "std_damage",

    "avg_damage_taken",

    "total_wins",

    "player_avg_kd",
    "player_max_kd",

    "player_avg_damage",
    "player_max_damage",

    "player_avg_kills",
    "player_avg_assists",
    "player_avg_kas",

    "twire_power",
    "twire_peak",
    "attacker_rating",
    "survivor_rating",
    "teammate_rating",
    "utility_rating",
    "finisher_rating",
    "star_players",
]

In [254]:
def train_model(history_features, features=FEATURES):
    X = history_features[features]
    y = history_features["avg_rank"]

    model = RandomForestRegressor(
        n_estimators=500,
        random_state=42
    )
    model.fit(X, y)

    return model, X, y


model, X, y = train_model(history_features)

In [255]:
def feature_importance(model, features=FEATURES):
    importance = pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_
    })

    return importance.sort_values(
        "importance",
        ascending=False
    )


feature_importance(model)

,feature,importance
10,total_wins,0.156968
4,std_rank,0.139549
1,stages,0.047396
12,player_max_kd,0.046579
25,star_players,0.044568
0,tournaments,0.044492
8,std_damage,0.043458
19,twire_peak,0.041920
3,std_points,0.041100
11,player_avg_kd,0.039029


In [256]:
def predict_ranks(model, ewc_df, features=FEATURES):
    ewc_df = ewc_df.copy()
    ewc_df["predicted_rank"] = model.predict(ewc_df[features])
    return ewc_df


ewc_df = predict_ranks(model, history_features)

In [257]:
def rank_predictions(ewc_df):
    return (
        ewc_df
        .sort_values("predicted_rank")
        [["team", "predicted_rank"]]
    )


prediction = rank_predictions(ewc_df)
prediction

,team,predicted_rank
10,SHADOW ESPORT,6.459090
22,VEGA ESPORTS,6.668151
4,Geekay Esports,6.899544
15,TYLOO,6.968155
14,TEAM NEMESIS,7.006471
12,Sharper Esport,7.164712
20,The Vicious,7.472593
13,T1,7.494985
9,Petrichor Road,7.530790
23,Virtus.pro,7.601059
